# EDA temporal - Dataset de consumo eléctrico

Este notebook se usa solo para exploración inicial. La implementación final de limpieza y transformación debe hacerse en Go dentro de `data-load/internal/dataset`.

Objetivo:
- Entender la estructura del dataset.
- Detectar valores faltantes o inválidos.
- Revisar rangos y outliers.
- Definir reglas de limpieza.
- Definir columnas transformadas útiles para ML.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)


## 1. Cargar una muestra del dataset

Ajusta la ruta según dónde abras el notebook. Si lo colocas en `/notebooks`, probablemente la ruta sea `../data/raw/household_power_consumption.txt`.


In [ ]:
DATA_PATH = Path('../data/raw/household_power_consumption.txt')

df = pd.read_csv(
    DATA_PATH,
    sep=';',
    nrows=200_000,
    low_memory=False
)

df.head()


## 2. Estructura básica

In [ ]:
print('Filas:', df.shape[0])
print('Columnas:', df.shape[1])
df.info()


In [ ]:
df.describe(include='all')

## 3. Detectar valores faltantes o inválidos

En este dataset es común encontrar `?` como valor faltante.


In [ ]:
invalid_counts = (df == '?').sum().sort_values(ascending=False)
invalid_counts


In [ ]:
missing_percent = ((df == '?').sum() / len(df) * 100).sort_values(ascending=False)
missing_percent


## 4. Convertir columnas numéricas para análisis

Esto es solo para exploración. En Go se debe implementar la misma regla con `strconv.ParseFloat`.


In [ ]:
numeric_cols = [
    'Global_active_power',
    'Global_reactive_power',
    'Voltage',
    'Global_intensity',
    'Sub_metering_1',
    'Sub_metering_2',
    'Sub_metering_3'
]

eda = df.copy()

for col in numeric_cols:
    eda[col] = pd.to_numeric(eda[col], errors='coerce')

eda[numeric_cols].isna().sum().sort_values(ascending=False)


## 5. Revisar rangos numéricos

In [ ]:
eda[numeric_cols].describe().T

## 6. Crear DateTime y variables temporales

In [ ]:
eda['DateTime'] = pd.to_datetime(
    eda['Date'] + ' ' + eda['Time'],
    format='%d/%m/%Y %H:%M:%S',
    errors='coerce'
)

eda['Hour'] = eda['DateTime'].dt.hour
eda['DayOfWeek'] = eda['DateTime'].dt.dayofweek
eda['Month'] = eda['DateTime'].dt.month

eda[['Date', 'Time', 'DateTime', 'Hour', 'DayOfWeek', 'Month']].head()


## 7. Crear variable de consumo no submedido

In [ ]:
eda['TotalActiveEnergyPerMinute'] = eda['Global_active_power'] * 1000 / 60

eda['SubMeteringTotal'] = (
    eda['Sub_metering_1'] +
    eda['Sub_metering_2'] +
    eda['Sub_metering_3']
)

eda['OtherConsumption'] = eda['TotalActiveEnergyPerMinute'] - eda['SubMeteringTotal']

eda[['Global_active_power', 'TotalActiveEnergyPerMinute', 'SubMeteringTotal', 'OtherConsumption']].head()


## 8. Analizar consumo por hora

In [ ]:
hourly = eda.groupby('Hour')['Global_active_power'].agg(['count', 'mean', 'median', 'max']).reset_index()
hourly


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(hourly['Hour'], hourly['mean'], marker='o')
plt.title('Consumo activo promedio por hora')
plt.xlabel('Hora del día')
plt.ylabel('Global active power promedio')
plt.grid(True)
plt.show()


## 9. Definir posible etiqueta para Machine Learning

In [ ]:
threshold_p75 = eda['Global_active_power'].quantile(0.75)
threshold_p90 = eda['Global_active_power'].quantile(0.90)

print('P75:', threshold_p75)
print('P90:', threshold_p90)

eda['HighConsumptionP75'] = (eda['Global_active_power'] >= threshold_p75).astype(int)
eda['HighConsumptionP90'] = (eda['Global_active_power'] >= threshold_p90).astype(int)

eda[['Global_active_power', 'HighConsumptionP75', 'HighConsumptionP90']].head()


## 10. Reglas finales sugeridas para implementar en Go

In [ ]:
summary = {
    'rows_sampled': len(df),
    'numeric_columns': numeric_cols,
    'invalid_question_mark_counts': invalid_counts.to_dict(),
    'missing_percent': missing_percent.to_dict(),
    'global_active_power_p75': float(threshold_p75),
    'global_active_power_p90': float(threshold_p90),
    'suggested_cleaning_rules': [
        'Descartar filas con columnas incompletas',
        'Descartar filas con ? en columnas numéricas',
        'Convertir columnas numéricas a float64',
        'Descartar filas con DateTime inválido',
        'Descartar filas con valores negativos en consumo o voltaje no positivo',
        'Crear Hour, DayOfWeek y Month desde DateTime',
        'Crear OtherConsumption = Global_active_power * 1000 / 60 - SubMeteringTotal',
        'Crear HighConsumption usando un umbral definido por percentil o regla de negocio'
    ]
}

summary
